In [16]:
with open("./data.txt", "r", encoding="utf-8") as f:
    text = f.read()
text[:15]

'hi everyone so '

In [17]:
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))

list(tokens[:15])

[104, 105, 32, 101, 118, 101, 114, 121, 111, 110, 101, 32, 115, 111, 32]

In [18]:
## Test of what zip does with ids

fake_tokens = [1, 2, 3, 4, 5, 6, 7, 8]

list(zip(fake_tokens, fake_tokens[1:]))

[(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8)]

In [19]:
def get_stats(ids):
    counts = {}
    for i, j in zip(ids, ids[1:]):
        counts[(i, j)] = counts.get((i, j), 0) + 1
    return counts

stats = get_stats(tokens)
sorted(stats.items(), key=lambda i: i[1], reverse=True)[:10]

[((101, 32), 5146),
 ((32, 116), 4903),
 ((116, 104), 3414),
 ((115, 32), 2996),
 ((116, 32), 2851),
 ((32, 97), 2647),
 ((105, 110), 2212),
 ((104, 101), 2204),
 ((32, 105), 1999),
 ((32, 115), 1920)]

In [20]:
max(stats, key=stats.get)

(101, 32)

In [21]:
 
def merge(ids, pair, idx):
    new_ids = []
    i = 0
    while i< len(ids):
        if i < len(ids)-1 and pair[0] == ids[i] and pair[1] == ids[i+1]:
            new_ids.append(idx)
            i+=2
        else:
            new_ids.append(ids[i])
            i+=1
    return new_ids


merge([4, 5, 6, 1, 5, 4, 4, 9, 6, 5, 4, 5, 4], (4, 5), 1000)

[1000, 6, 1, 5, 4, 4, 9, 6, 5, 1000, 4]

In [22]:
vocab_size = 276
num_merges = vocab_size - 256

ids = list(tokens)

merges = {}

tok = 256

for _ in range(num_merges):
    stats = get_stats(ids)
    max_tok = max(stats, key=stats.get)
    print(f"Merging {max_tok} into new token {tok}")
    ids = merge(ids, max_tok, tok)
    merges[max_tok] = tok
    tok += 1

Merging (101, 32) into new token 256
Merging (32, 116) into new token 257
Merging (257, 104) into new token 258
Merging (115, 32) into new token 259
Merging (116, 32) into new token 260
Merging (105, 110) into new token 261
Merging (101, 110) into new token 262
Merging (97, 110) into new token 263
Merging (101, 114) into new token 264
Merging (100, 32) into new token 265
Merging (111, 32) into new token 266
Merging (121, 32) into new token 267
Merging (261, 103) into new token 268
Merging (258, 256) into new token 269
Merging (105, 259) into new token 270
Merging (111, 110) into new token 271
Merging (111, 117) into new token 272
Merging (116, 104) into new token 273
Merging (111, 114) into new token 274
Merging (32, 115) into new token 275


In [23]:
print("Original token list length: ", len(tokens))
print("New token list length: ", len(ids))
print("Compression ratio: %.2fX" % (len(tokens) / len(ids)))

Original token list length:  127302
New token list length:  92951
Compression ratio: 1.37X


In [28]:
# This is my first attempt at building the decode function

def find_next(ids, iD):
    expanded_ids = []
    next_merge = next(
        (key for key, value in merges.items() if value == iD),
        None
    )
    for nid in next_merge:
        if nid > 255:
            expanded_ids += find_next(ids, nid)
        else:
            expanded_ids.append(nid)
    return expanded_ids

def decode(ids):
    decoded_ids = []
    for iD in ids:
        if iD <= 255:
            decoded_ids.append(iD)
        else:
            decoded_ids = decoded_ids + find_next(ids, iD)
    return decoded_ids
            

print(ids[:20])
print(decode(ids[:20]))
bytes(decode(ids[:20])).decode("utf-8", errors="replace")

[104, 105, 32, 101, 118, 264, 121, 271, 256, 115, 266, 261, 258, 270, 118, 105, 100, 101, 266, 73]
[104, 105, 32, 101, 118, 101, 114, 121, 111, 110, 101, 32, 115, 111, 32, 105, 110, 32, 116, 104, 105, 115, 32, 118, 105, 100, 101, 111, 32, 73]


'hi everyone so in this video I'

When I saw Adrej's code I realized I did'nt have to make a recursive function to get the bytes, the list is already sorted so I can just build a vocab dictionary and it is guaranteed (at least in new versions of python) that the vocab will be bilt with valid bytes since we start by the first merge and so on.

In [29]:
# This is the basic vocab
vocab = {idx: bytes([idx]) for idx in range(256)}

# Every next merge builts on top of valid bytes
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

print(decode(ids[:20]))

hi everyone so in this video I


Encoding: Is taking each pair of the sequence to encode and replace it with the most eligible merge until there is no more valid merges you can make

In [30]:

text_to_encode = "Decoding: Is taking each pair of the sequence to encode and replace it with the most eligible merge until there is no more valid merges you can make"


def encode(text):
    tokens = list(text.encode("utf-8"))

    while len(tokens) >= 2:
        pairs = get_stats(tokens)
        pair = min(pairs, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    
    return tokens

encode("this is the tokenizer")


[273, 270, 105, 115, 269, 116, 111, 107, 262, 105, 122, 264]